# Практика: базовый RAG-pipeline (Jupyter Notebook)

Цель: собрать минимальный Retrieval-Augmented Generation (RAG) пайплайн для ответов на вопросы **по вашим документам**.

Что сделаем:
1) загрузим документы (txt/markdown или PDF),  
2) разобьём на чанки,  
3) построим эмбеддинги,  
4) проиндексируем во векторном хранилище (FAISS локально),  
5) реализуем retrieval + генерацию ответа LLM с цитированием источников.

> Этот ноутбук самодостаточен: можно запустить локально.  
> Для LLM/эмбеддингов есть два режима:
- **OpenAI** (нужен ключ в переменной окружения `OPENAI_API_KEY`)
- **Локальные эмбеддинги** через `sentence-transformers` (без ключа)

## 0) Установка зависимостей

Выполните одну из команд (в зависимости от ваших предпочтений).

**Вариант A (рекомендовано для старта):** LangChain + FAISS + sentence-transformers + PDF

In [1]:
# Если вы запускаете в чистом окружении:
%pip install -U langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf

# Если планируете использовать OpenAI:
%pip install -U langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
  Attempting uninstall

## 1) Импорт и конфигурация

Выберите провайдера эмбеддингов и LLM.

- `USE_OPENAI = True` - если есть ключ OpenAI
- `USE_OPENAI = False` - если хотите локальные эмбеддинги и (опционально) локальную LLM

In [2]:
import os
from typing import List, Dict, Any
from pathlib import Path

USE_OPENAI = True  # <- поставьте True, если хотите OpenAI

### 1.1) Провайдер эмбеддингов

In [3]:
import os # Import os module
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

if USE_OPENAI:
    from langchain_openai import OpenAIEmbeddings
    # Set the OPENAI_API_KEY as an environment variable before initializing OpenAIEmbeddings
    # The variable OPENAI_API_KEY is already loaded in the notebook's kernel state

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

### 1.2) Провайдер LLM

Если `USE_OPENAI=True`, используем OpenAI Chat model.
Если `USE_OPENAI=False`, используем заглушку, чтобы пайплайн retrieval был runnable и без LLM.

In [5]:
if USE_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
else:
    llm = None  # заглушка ниже

## 2) Подготовка документов

Добавьте свои `.txt`/`.md`/`.pdf` файлы в папку `./data/`.

Если папки `data/` нет - создадим и положим демо-документ.

In [6]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

demo_path = DATA_DIR / "demo_policy.md"
if not demo_path.exists():
    demo_path.write_text(
        "# Политика отпусков (пример)\n"
        "Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.\n"
        "Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.\n"
        "Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.\n",
        encoding="utf-8"
    )

print("Файлы в ./data:")
for p in sorted(DATA_DIR.glob("*")):
    print(" -", p.name)

Файлы в ./data:
 - demo_policy.md


### 2.1) Загрузка документов

In [7]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_core.documents import Document # Corrected import path for Document

def load_documents(data_dir: Path) -> List[Document]:
    docs: List[Document] = []
    for path in data_dir.glob("*"):
        suf = path.suffix.lower()
        if suf in [".txt", ".md"]:
            docs.extend(TextLoader(str(path), encoding="utf-8").load())
        elif suf == ".pdf":
            docs.extend(PyPDFLoader(str(path)).load())
    return docs

docs = load_documents(DATA_DIR)
print(f"Загружено документов: {len(docs)}")
print("Пример:", docs[0].metadata)
print(docs[0].page_content[:400])

Загружено документов: 1
Пример: {'source': 'data/demo_policy.md'}
# Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.



## 3) Chunking (разбиение на фрагменты)

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print(f"Чанков: {len(chunks)}")
print("Пример чанка:\n", chunks[0].page_content[:400])

Чанков: 1
Пример чанка:
 # Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.


## 4) Индексация во векторное хранилище (FAISS)

In [9]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# (опционально) сохранить индекс:
vectorstore.save_local("faiss_index")

print("Готово: индекс построен.")

Готово: индекс построен.


## 5) Retrieval: проверим, что поиск работает

In [10]:
def pretty_docs(docs: List[Any]) -> None:
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        where = f"{src}" + (f" (page {page})" if page is not None else "")
        print(f"\n--- [#{i}] {where} ---")
        print(d.page_content[:800])

question = "Сколько дней ежегодного оплачиваемого отпуска?"
hits = retriever.invoke(question) # Changed from get_relevant_documents

pretty_docs(hits)


--- [#1] data/demo_policy.md ---
# Политика отпусков (пример)
Компания предоставляет ежегодный оплачиваемый отпуск 35 календарных дней.
Сотрудник может брать отпуск частями, но одна часть должна быть не менее 10 дней.
Для оформления отпуска нужно подать заявление не позднее чем за 30 дней до начала.


## 6) Генерация ответа без RAG

In [13]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = (
    "Ты - ассистент, который отвечает на вопросы строго по предоставленному контексту.\n"
    "Правила:\n"
    "1) Не выдумывай факты.\n"
    "2) В конце добавь раздел 'Источники:' и перечисли источники (source/page).\n"
    "3) Если в источниках нет информации в разедел 'Источники:' добавь 'Базовая модель'"
)

def no_rag_answer(question: str) -> Dict[str, Any]:
    if USE_OPENAI:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Вопрос: {question}")
        ]
        resp = llm.invoke(messages)
        answer_text = resp.content
    else:
        answer_text = (
            "LLM не подключена (USE_OPENAI=False)")

    return {"question": question, "answer": answer_text}

result = no_rag_answer("Какая минимальная длина одной части отпуска?")
print(result["answer"])

Минимальная длина одной части отпуска составляет 14 календарных дней. 

Источники: Базовая модель


## 7) Генерация ответа (RAG)

Соберём prompt так, чтобы модель отвечала строго по контексту и добавляла источники.

Если `USE_OPENAI=False`, вместо LLM вернём релевантный контекст (заглушка).

In [14]:
SYSTEM_PROMPT = (
    "Ты - ассистент, который отвечает на вопросы строго по предоставленному контексту.\n"
    "Правила:\n"
    "1) Если в контексте нет ответа - скажи: 'В документах нет ответа на этот вопрос.'\n"
    "2) Не выдумывай факты.\n"
    "3) В конце добавь раздел 'Источники:' и перечисли источники (source/page).\n"
)

def build_context(docs: List[Any]) -> str:
    parts = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        tag = f"{src}" + (f":{page}" if page is not None else "")
        parts.append(f"[{tag}]\n{d.page_content}")
    return "\n\n".join(parts)

def rag_answer(question: str) -> Dict[str, Any]:
    retrieved = retriever.invoke(question) # Changed from get_relevant_documents
    context = build_context(retrieved)

    if USE_OPENAI:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Вопрос: {question}\n\nКонтекст:\n{context}")
        ]
        resp = llm.invoke(messages)
        answer_text = resp.content
    else:
        answer_text = (
            "LLM не подключена (USE_OPENAI=False). Ниже релевантный контекст, который можно подать в LLM:\n\n"
            + context[:1200]
            + ("\n\n... (контекст обрезан)" if len(context) > 1200 else "")
        )

    sources = [{"source": d.metadata.get("source", "unknown"), "page": d.metadata.get("page", None)} for d in retrieved]
    return {"question": question, "answer": answer_text, "sources": sources, "retrieved_docs": retrieved}

result = rag_answer("Какая минимальная длина одной части отпуска?")
print(result["answer"])

Минимальная длина одной части отпуска составляет 10 дней.

Источники:
source/data/demo_policy.md


## 7) Удобная функция для вопросов

In [18]:
def ask(q: str):
    res = rag_answer(q)
    print("\n=== Вопрос ===\n", res["question"])
    print("\n=== Ответ ===\n", res["answer"])

ask("За сколько дней до начала отпуска нужно подать заявление?")


=== Вопрос ===
 За сколько дней до начала отпуска нужно подать заявление?

=== Ответ ===
 Заявление на отпуск нужно подать не позднее чем за 30 дней до начала. 

Источники:
source/page: data/demo_policy.md
